# Creating a Daily Master Table



Load in the exisiting parquet data sets into individual pandas dataframes to be comined into one daily master table, which will become the first master dataset for this project. \
Note: Some additional cleaning might have to be done in order to have each row indicate a particular day.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
data_path = Path.cwd() / 'data_processed'

acute_training_load_path = data_path / 'acute_training_load_v1.parquet'
max_met_path = data_path / 'max_met_v1.parquet'
race_predictions_path = data_path / 'race_predicitons_v1.parquet'
run_path = data_path / 'runs_v1.parquet'
training_history_path = data_path / 'training_history_v1.parquet'
training_readiness_path = data_path / 'training_readiness_v1.parquet'

In [3]:
# Create a daily runs data frame 
runs_df = pd.read_parquet(run_path)

runs_df["date"] = runs_df["start_time"].dt.normalize()

# Merge by day, using only features that make sense to aggregate daily
daily_runs_df = (
    runs_df
    .groupby("date")
    .agg(
        run_count=("activityId", "count"),
        total_distance_km=("distance_km", "sum"),
        total_distance_miles=("distance_miles", "sum"),
        total_duration_minutes=("duration_minutes", "sum"),
        total_moving_minutes=("moving_duration_minutes", "sum"),
        total_elevation_gain_m=("elevation_gain_m", "sum"),
        avg_hr=("avgHr", "mean"),
        max_hr=("maxHr", "max"),
        avg_pace_mile=("avg_speed_mps", lambda x: 26.8224 / x.mean()),  # min/mi
        avg_power=("avgPower", "mean"),
        max_power=("maxPower", "max"),
        avg_cadence=("avgDoubleCadence", "mean"),
        avg_stride_length_m=("avg_stride_length_m", "mean"),
        total_training_load=("activityTrainingLoad", "sum"),
        aerobic_te_avg=("aerobicTrainingEffect", "mean"),
        anaerobic_te_avg=("anaerobicTrainingEffect", "mean"),
        pr_count=("pr", "sum"),
    )
    .reset_index()
)

print(daily_runs_df.sample(3))
print(daily_runs_df.shape)

# Show me wht columns have the most amount of missing data
daily_runs_df.isna().mean().sort_values(ascending=False)



          date  run_count  total_distance_km  total_distance_miles  \
616 2026-02-05          1           8.074880              5.017498   
57  2023-01-08          1           8.291660              5.152199   
389 2024-12-23          1          21.012471             13.056544   

     total_duration_minutes  total_moving_minutes  total_elevation_gain_m  \
616               41.718335             41.573700                    64.0   
57                50.009200             49.930467                    73.0   
389               95.003402             93.988567                    60.0   

     avg_hr  max_hr  avg_pace_mile  avg_power  max_power  avg_cadence  \
616   147.0   165.0       8.314445      242.0      327.0     171.2500   
57    176.0   191.0       9.707709        NaN        NaN     160.8125   
389   164.0   179.0       7.276831      262.0      397.0     166.8125   

     avg_stride_length_m  total_training_load  aerobic_te_avg  \
616             1.124300            70.039703       

avg_power                 0.442815
max_power                 0.442815
avg_stride_length_m       0.026393
anaerobic_te_avg          0.021994
aerobic_te_avg            0.021994
avg_cadence               0.021994
avg_hr                    0.021994
max_hr                    0.021994
date                      0.000000
total_training_load       0.000000
avg_pace_mile             0.000000
run_count                 0.000000
total_elevation_gain_m    0.000000
total_moving_minutes      0.000000
total_duration_minutes    0.000000
total_distance_miles      0.000000
total_distance_km         0.000000
pr_count                  0.000000
dtype: float64

In [4]:
daily_runs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    682 non-null    datetime64[ns]
 1   run_count               682 non-null    int64         
 2   total_distance_km       682 non-null    float64       
 3   total_distance_miles    682 non-null    float64       
 4   total_duration_minutes  682 non-null    float64       
 5   total_moving_minutes    682 non-null    float64       
 6   total_elevation_gain_m  682 non-null    float64       
 7   avg_hr                  667 non-null    float64       
 8   max_hr                  667 non-null    float64       
 9   avg_pace_mile           682 non-null    float64       
 10  avg_power               380 non-null    float64       
 11  max_power               380 non-null    float64       
 12  avg_cadence             667 non-null    float64   

In [5]:
# Load in acute training load
acute_training_load_df = pd.read_parquet(acute_training_load_path)

# Select only the relevant columns
acute_subset = acute_training_load_df[[
    "acwrPercent",
    "acwrStatus",
    "acwrStatusFeedback",
    "dailyTrainingLoadAcute",
    "dailyTrainingLoadChronic",
    "dailyAcuteChronicWorkloadRatio",
    "date"

]]

# Create a daily master table
daily_master = pd.merge(
    daily_runs_df,
    acute_subset,
    on='date',
    how='left'
)


In [6]:
# Load 
daily_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   date                            682 non-null    datetime64[ns]
 1   run_count                       682 non-null    int64         
 2   total_distance_km               682 non-null    float64       
 3   total_distance_miles            682 non-null    float64       
 4   total_duration_minutes          682 non-null    float64       
 5   total_moving_minutes            682 non-null    float64       
 6   total_elevation_gain_m          682 non-null    float64       
 7   avg_hr                          667 non-null    float64       
 8   max_hr                          667 non-null    float64       
 9   avg_pace_mile                   682 non-null    float64       
 10  avg_power                       380 non-null    float64       
 11  max_po

In [7]:
# Load race predictions
race_predictions_df = pd.read_parquet(race_predictions_path)

print(race_predictions_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 693 entries, 0 to 692
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   userProfilePK     693 non-null    int64 
 1   calendarDate      693 non-null    object
 2   deviceId          693 non-null    int64 
 3   timestamp         693 non-null    object
 4   raceTime5K        693 non-null    int64 
 5   raceTime10K       693 non-null    int64 
 6   raceTimeHalf      693 non-null    int64 
 7   raceTimeMarathon  693 non-null    int64 
 8   date              693 non-null    object
 9   5K_pred           693 non-null    object
 10  10K_pred          693 non-null    object
 11  Half_pred         693 non-null    object
 12  Marathon_pred     693 non-null    object
dtypes: int64(6), object(7)
memory usage: 70.5+ KB
None


In [8]:
# Create a subset of the relevant columns
race_prediction_subset = race_predictions_df[[
    "date",
    "5K_pred",
    "10K_pred",
    "Half_pred",
    "Marathon_pred"
]]

race_prediction_subset["date"] = pd.to_datetime(race_prediction_subset['date'])

# Merge with daily master
daily_master = pd.merge(
    daily_master,
    race_prediction_subset,
    on='date',
    how='left'
)

/var/folders/5p/042l2mfn5lg_ltr5w11xxs9h0000gn/T/ipykernel_16025/1913496268.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race_prediction_subset["date"] = pd.to_datetime(race_prediction_subset['date'])


In [9]:
# Load Max Met Data
max_met_df = pd.read_parquet(max_met_path)


max_met_df['date'] = pd.to_datetime(max_met_df['calendarDate'])
max_met_subset = max_met_df[[
    "vo2MaxValue",
    "fitnessAge",
    "fitnessAgeDescription",
    "maxMet",
    "date",
]]

# Merge with daily master
daily_master = pd.merge(
    daily_master,
    max_met_subset,
    on='date',
    how='left'
)

In [10]:
print(daily_master.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 753 entries, 0 to 752
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   date                            753 non-null    datetime64[ns]
 1   run_count                       753 non-null    int64         
 2   total_distance_km               753 non-null    float64       
 3   total_distance_miles            753 non-null    float64       
 4   total_duration_minutes          753 non-null    float64       
 5   total_moving_minutes            753 non-null    float64       
 6   total_elevation_gain_m          753 non-null    float64       
 7   avg_hr                          738 non-null    float64       
 8   max_hr                          738 non-null    float64       
 9   avg_pace_mile                   753 non-null    float64       
 10  avg_power                       422 non-null    float64       
 11  max_po

In [11]:
# Load training readiness
training_readiness_df = pd.read_parquet(training_readiness_path)

training_readiness_events_df = training_readiness_df.copy()

training_readiness_daily_df = (
    training_readiness_df
    .sort_values(["calendarDate", "timestampLocal"])
    .groupby("calendarDate")
    .agg(
        readiness_score_first=("score", "first"),
        readiness_score_last=("score", "last"),
        readiness_score_mean=("score", "mean"),
        readiness_level_last=("level", "last"),
        recovery_time_last=("recoveryTime", "last"),
        acute_load_last=("acuteLoad", "last"),
        hrv_weekly_avg_last=("hrvWeeklyAverage", "last"),
        sleep_score_last=("sleepScore", "last"),
        valid_sleep_any=("validSleep", "max"),
        readiness_snapshots=("score", "count"),
    )
    .reset_index()
)


training_readiness_daily_df['date'] = pd.to_datetime(training_readiness_daily_df['calendarDate'])
training_readiness_daily_df = training_readiness_daily_df.drop(columns=[
    "calendarDate"
])

print(training_readiness_daily_df.info())


# Merge with daily master
daily_master = pd.merge(
    daily_master,
    training_readiness_daily_df,
    on='date',
    how='left'
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 563 entries, 0 to 562
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   readiness_score_first  559 non-null    float64       
 1   readiness_score_last   559 non-null    float64       
 2   readiness_score_mean   559 non-null    float64       
 3   readiness_level_last   563 non-null    object        
 4   recovery_time_last     563 non-null    int64         
 5   acute_load_last        563 non-null    int64         
 6   hrv_weekly_avg_last    558 non-null    float64       
 7   sleep_score_last       334 non-null    float64       
 8   valid_sleep_any        563 non-null    bool          
 9   readiness_snapshots    563 non-null    int64         
 10  date                   563 non-null    datetime64[ns]
dtypes: bool(1), datetime64[ns](1), float64(5), int64(3), object(1)
memory usage: 44.7+ KB
None


In [12]:
# Load training history
training_history_df = pd.read_parquet(training_history_path)

print(training_history_df.info())
print(training_history_df['calendarDate'].duplicated().sum())

training_history_daily_df = (
    training_history_df
    .sort_values(["calendarDate"])
    .groupby("calendarDate")
    .agg(
        load_tunnel_min=("loadTunnelMin", "min"),
        load_tunnel_max=("loadTunnelMax", "max"),
        training_status_first=("trainingStatus", "first"),
        training_status_last=("trainingStatus", "last"),
        training_status=("trainingStatus", "last"),
        fitness_level_trend=("fitnessLevelTrend", "last"),
        load_level_trend=("loadLevelTrend", "last")
    )
    .reset_index()
)

training_history_daily_df['date'] = pd.to_datetime(training_history_daily_df['calendarDate'])
training_history_daily_df = training_history_daily_df.drop(columns='calendarDate')


# Merge with daily master
daily_master = pd.merge(
    daily_master,
    training_history_daily_df,
    on='date',
    how='left'
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2725 entries, 0 to 2724
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   calendarDate                   2725 non-null   object 
 1   timestamp                      2725 non-null   object 
 2   sport                          2706 non-null   object 
 3   subSport                       2706 non-null   object 
 4   weeklyTrainingLoadSum          1290 non-null   float64
 5   loadTunnelMin                  1290 non-null   float64
 6   loadTunnelMax                  1290 non-null   float64
 7   trainingStatus                 2725 non-null   object 
 8   fitnessLevelTrend              2725 non-null   object 
 9   loadLevelTrend                 1290 non-null   object 
 10  trainingStatus2FeedbackPhrase  1435 non-null   object 
dtypes: float64(3), object(8)
memory usage: 234.3+ KB
None
1377


In [13]:
print(daily_master.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 753 entries, 0 to 752
Data columns (total 49 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   date                            753 non-null    datetime64[ns]
 1   run_count                       753 non-null    int64         
 2   total_distance_km               753 non-null    float64       
 3   total_distance_miles            753 non-null    float64       
 4   total_duration_minutes          753 non-null    float64       
 5   total_moving_minutes            753 non-null    float64       
 6   total_elevation_gain_m          753 non-null    float64       
 7   avg_hr                          738 non-null    float64       
 8   max_hr                          738 non-null    float64       
 9   avg_pace_mile                   753 non-null    float64       
 10  avg_power                       422 non-null    float64       
 11  max_po

In [15]:
# Save the daily master table to parquet
daily_master.to_parquet(
    "../data_processed/daily_master_v1.parquet",
    index=False
)